# Module 2 Hydrologic Connectivity
**Region:** Cache la Poudre watershed/2020 Cameron Peak Fire burn scar

Module 3 computes erosion susceptibility per cell. That answers "could sediment
move here?" but not "does it get anywhere?"and for a water provider on the
Poudre, the second question is the one with a dollar figure attached.
This module routes water. 

- **Real sub-watersheds.** NHD/WBD HU12 polygons replace the illustrative
  grid-based split used in `wildfire-landscape-intelligence` Module 4. Agency
  partners already work in HUCs; a grid split cannot be handed to anyone.
- **Actual flow routing.** `np.gradient` slope is a per-cell property.
  Connectivity is path-dependent, so it needs D8 routing over the whole grid.
  That is what the new `daear_toolkit.hydrology` module provides pure
  NumPy/SciPy, no GDAL build chain.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
import daear_toolkit as dt
from daear_toolkit import data_access, indicators, viz, hydrology

REGION = dt.POUDRE_CAMERON_PEAK
BBOX = REGION.bbox

terrain = data_access.get_terrain(BBOX)
burn    = data_access.get_burn_severity(BBOX, fire_year=REGION.fire_year)

dem = terrain["elevation_m"]
print(f"DEM: {dem.shape}, {float(dem.min()):.0f}-{float(dem.max()):.0f} m elevation")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
viz.plot_raster(dem,  title="Elevation (m)",                          ax=axes[0], cmap="terrain")
viz.plot_raster(burn, title="Cameron Peak burn severity (2020)",       ax=axes[1], cmap=viz.SEVERITY_CMAP)
plt.tight_layout()
plt.savefig("../outputs/02_inputs.png", dpi=150)
plt.show()

In [ ]:
# Check the DEM 
dem = terrain["elevation_m"]
print(f"DEM: {dem.shape}, {float(dem.min()):.0f}-{float(dem.max()):.0f} m elevation")

print("Terrain variables:", list(terrain.data_vars))

## Fill depressions, then route flow

Depression filling is not optional. Real DEMs are full of pits 
sensor noise, culverts under roads, genuine closed basins and routing
on an unfilled DEM produces a broken, stubby drainage network because water
gets trapped in every one of them.

`hydrology.fill_depressions` runs priority-flood: seed a min-heap with the
raster edge, then repeatedly flood outward from the lowest unprocessed cell.
Because expansion always happens from the lowest available point, each interior
cell gets raised exactly to the lowest spill elevation on any path to the edge.

Then D8 routing sends each cell's water to whichever neighbour offers the
steepest drop per unit distance, and accumulation sums contributors in
descending-elevation order with one pass, no graph library needed.

In [ ]:
filled = hydrology.fill_depressions(dem)
fill_depth = filled - dem

print(f"Cells modified by filling: {float((fill_depth > 0.01).mean()):.2%}")
print(f"Deepest pit filled:        {float(fill_depth.max()):.1f} m")

direction = hydrology.flow_direction(filled, prefilled=True)
acc = hydrology.flow_accumulation(dem)

# Mass-balance check. Every cell's water has to leave through some outlet, so
# accumulation summed over outlet cells must equal the valid cell count. If this
# fails, the routing is dropping water somewhere and nothing downstream is trustworthy.
outlets = direction < 0
total_out = float(acc.where(outlets).sum())
n_valid = int(np.isfinite(dem.values).sum())
print(f"Mass balance: {total_out:.0f} accumulated at outlets vs. {n_valid} valid cells "
      f"({'OK' if abs(total_out - n_valid) < 1 else 'MISMATCH -- investigate'})")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
viz.plot_raster(fill_depth.where(fill_depth > 0.01), title="Depression fill depth (m)", ax=axes[0], cmap="Blues")
viz.plot_raster(np.log10(acc.where(acc > 0)), title="Flow accumulation (log10 cells)", ax=axes[1], cmap="Blues")
plt.tight_layout()
plt.savefig("../outputs/02_flow_accumulation.png", dpi=150)
plt.show()

## Extract streams, and check them against NHD

The accumulation threshold that defines a "channel" is the classic
arbitrary-but-tunable parameter. Rather than defending a number, validate it:
pull real NHD flowlines and tune until the derived network has comparable
density. That takes the choice out of the analyst's hands, which is what makes
it defensible to a reviewer.

In [ ]:
STREAM_THRESHOLD = 500  # cells; at 30 m this is ~0.45 km2 of contributing area

streams = hydrology.extract_streams(acc, threshold=STREAM_THRESHOLD)
flowlines = data_access.get_flowlines(BBOX, min_order=2)   # NHD, order>=2 drops ephemeral clutter

fig, ax = plt.subplots(figsize=(8, 6.5))
viz.plot_raster(streams.where(streams > 0), title="Derived streams (blue) vs. NHD flowlines (black)", ax=ax, cmap="Blues")
flowlines.plot(ax=ax, color="black", linewidth=0.6, alpha=0.8)
plt.tight_layout()
plt.savefig("../outputs/02_stream_validation.png", dpi=150)
plt.show()

derived_km = float(streams.sum()) * 30/1000
nhd_km = float(flowlines.to_crs(epsg=5070).length.sum()) / 1000
print(f"Derived channel length: {derived_km:.0f} km")
print(f"NHD flowline length:    {nhd_km:.0f} km")
print(f"Ratio: {derived_km / nhd_km:.2f}  (aim for 0.8-1.2; raise the threshold if too high)")

## Sub-watersheds from WBD

HU12 units run roughly 40–160 km². That is the right grain for post-fire work:
small enough that the burn scar dominates some units and barely touches others,
which is exactly the contrast that makes the ranking later in this notebook meaningful.

If WBD coverage were unavailable, `hydrology.subwatershed_labels` derives basins
from the routing instead. Prefer the polygons when you can get them if partners
already have those unit IDs in their own systems.

In [ ]:
huc12 = data_access.get_watershed_boundaries(BBOX, level=12, clip=True)
print(f"{len(huc12)} HU12 sub-watersheds intersect the AOI")

watershed_id, huc_lookup = data_access.rasterize_watersheds(huc12, template=dem)

fig, ax = plt.subplots(figsize=(8, 6.5))
viz.plot_raster(watershed_id, title="HU12 sub-watersheds", ax=ax, cmap="tab20")
huc12.boundary.plot(ax=ax, color="black", linewidth=0.5)
plt.tight_layout()
plt.savefig("../outputs/02_subwatersheds.png", dpi=150)
plt.show()

huc12[["huc12", "name"]].head(10)

## The connectivity index

`hydrology.sediment_connectivity_index` implements the Borselli/Cavalli
Index of Connectivity:

$$IC = \log_{10}\left(\frac{D_{up}}{D_{dn}}\right), \qquad
D_{up} = \bar{W}\bar{S}\sqrt{A}, \qquad
D_{dn} = \sum_i \frac{d_i}{W_i S_i}$$

$W$ is the transfer weight or how readily a cell passes runoff downslope. In the
original formulation it comes from the USLE cover factor. The post-fire
adaptation, and the reason the Cameron Peak scar shows up here as a genuine
anomaly rather than a slope artefact, is deriving $W$ from burn severity and
vegetation cover instead.

In [ ]:
ndvi = indicators.ndvi(
    data_access.get_optical_scene(BBOX, date="2024-08-15", source="sentinel-2")
)

W = indicators.transfer_weight(burn_severity=burn, ndvi=ndvi)
ic = hydrology.sediment_connectivity_index(dem, terrain["slope_deg"], impedance=W, streams=streams)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
viz.plot_raster(W, title="Transfer weight W (higher = less impedance)", ax=axes[0], cmap="YlOrBr")
viz.plot_raster(ic, title="Index of connectivity (log10)", ax=axes[1], cmap=viz.SEVERITY_CMAP)
plt.tight_layout()
plt.savefig("../outputs/02_connectivity_index.png", dpi=150)
plt.show()

burned = burn > 0.5
print(f"Mean IC, high-severity burn: {float(ic.where(burned).mean()):.2f}")
print(f"Mean IC, unburned:           {float(ic.where(~burned).mean()):.2f}")
print("  -> the gap between these is the fire's hydrologic signature, not its thermal one")

## Delivery risk, ranked by sub-watershed

Connectivity alone is not the answer either. Erodible ground on a disconnected
bench is a soil problem; the same ground directly coupled to a channel is a
reservoir-turbidity problem. `indicators.delivery_risk` multiplies the two
rather than averaging them, because sediment that cannot move, or cannot arrive, 
does not reach the intake.

Flow-path distance is worth noting as distinct from Euclidean distance: a cell
50 m from a stream but on the wrong side of a divide is hundreds of metres away
in routing terms, and it is routing that governs delivery.

In [ ]:
erosion = indicators.erosion_susceptibility(terrain["slope_deg"], burn, soil_runoff := data_access.get_soil_properties(BBOX)["runoff_potential"])
risk = indicators.delivery_risk(ic, erosion)
dist = hydrology.distance_to_stream(dem, streams)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
viz.plot_raster(dist, title="Flow-path distance to channel (m)", ax=axes[0], cmap="viridis_r")
viz.plot_raster(risk, title="Sediment delivery risk", ax=axes[1], cmap=viz.SEVERITY_CMAP)
plt.tight_layout()
plt.savefig("../outputs/02_delivery_risk.png", dpi=150)
plt.show()

In [ ]:
rows = []
for code, huc in huc_lookup.items():
    mask = watershed_id == code
    if float(mask.sum()) < 100:
        continue
    rows.append({
        "huc12": huc,
        "name": huc12.loc[huc12["huc12"] == huc, "name"].squeeze() if "name" in huc12 else "",
        "area_cells": int(mask.sum()),
        "pct_high_severity": float((burn.where(mask) > 0.5).mean()) * 100,
        "mean_connectivity": float(ic.where(mask).mean()),
        "mean_delivery_risk": float(risk.where(mask).mean()),
    })

ranked = pd.DataFrame(rows).sort_values("mean_delivery_risk", ascending=False).round(3)
ranked.to_csv("../outputs/02_subwatershed_delivery_risk.csv", index=False)
print("Top 8 sub-watersheds by sediment delivery risk:")
ranked.head(8)

## Summary

Fire severity, terrain, and vegetation routed into a per-cell connectivity
index, then rolled up to real HU12 units and ranked.

The ranked table is the deliverable. It converts "the Cameron Peak scar is
large and severe" into "these specific sub-watersheds are the ones wired
directly into the mainstem, in this order" which is a treatment-prioritization
and monitoring-siting answer, addressed to units that partners already manage in.

**Known limitations:**

- D8 routing exaggerates flow convergence on planar hillslopes relative to
  D-infinity or MFD. Fine at the burn-scar-to-mainstem scale asked about here;
  not fine for hillslope-scale sediment budgets.
- IC is a relative index. Compare within this region, not across regions with
  different cell sizes or different $W$ derivations.
- Neither IC nor delivery risk is a sediment yield. Converting to tonnes needs
  calibration against gauge or reservoir sedimentation records, which the Poudre
  has and that is the obvious next step if a partner wants absolute numbers.